In [1]:
%load_ext autoreload
%autoreload 2

In [50]:
# Libraries
import pandas as pd

# =====================================================================
# 2. SYNTHETIC TEST DATA
# =====================================================================

# Problems table (SNOMED exact codes + multiple occurrences)
df_problems = pd.DataFrame(
    [
        { "SUBJECT": 10001, "PROBLEM_CODE": "84089009",  "PROBLEM_DT_TM": "2020-08-02" },
        { "SUBJECT": 10001, "PROBLEM_CODE": "235494005", "PROBLEM_DT_TM": "2020-12-02" },
        { "SUBJECT": 10001, "PROBLEM_CODE": "235494005", "PROBLEM_DT_TM": "2020-10-01" },
        { "SUBJECT": 10002, "PROBLEM_CODE": "71457002",  "PROBLEM_DT_TM": "2021-11-12" },
    ]
)

# Diagnoses table (ICD-10 formatting variations: standard, 1 dot, 2 dots)
df_diagnoses = pd.DataFrame(
    [
        { "SUBJECT": 10001, "DIAGNOSIS_CODE_ICD": "R41.3",   "DIAGNOSIS_DATE": "2020-10-31" },
        { "SUBJECT": 10002, "DIAGNOSIS_CODE_ICD": "M79.60",  "DIAGNOSIS_DATE": "2023-05-21" },
        { "SUBJECT": 10003, "DIAGNOSIS_CODE_ICD": "M7960",   "DIAGNOSIS_DATE": "2023-06-01" },
        { "SUBJECT": 10004, "DIAGNOSIS_CODE_ICD": "H18.4.1", "DIAGNOSIS_DATE": "2021-10-06" },
        { "SUBJECT": 10005, "DIAGNOSIS_CODE_ICD": "R413", "DIAGNOSIS_DATE": "2022-01-01" },
    ]
)

# Prescriptions table (Medication text searching)
df_prescriptions = pd.DataFrame(
    [
        { "SUBJECT": 10001, "MEDICATION_NAME": "Amoxicillin 500mg", "ORDER_DT_TM": "2022-05-27" },
        { "SUBJECT": 10001, "MEDICATION_NAME": "Lisinopril 10mg ", "ORDER_DT_TM": "2023-04-13" },
        { "SUBJECT": 10002, "MEDICATION_NAME": "Propofol 10mg/ml", "ORDER_DT_TM": "2022-05-07"},
    ]
)

In [51]:
# =====================================================================
# 3. TEST RUNNER WITH ASSERTIONS
# =====================================================================
# Libraries
from icare_risk.clinphen.utils.vectorized import build_historical_events_table

def run_tests():
    print("--- Running Test Bench ---\n")

    # TEST 1: Exact SNOMED matching and First Occurrence Aggregation
    configs_snomed = {
        "charlson_pancreatitis": {"codes": ["235494005"]},
        "charlson_hernia": {"codes": ["84089009"]},
    }

    res1 = build_historical_events_table(
        df=df_problems,
        configs=configs_snomed,
        subject_col="SUBJECT",
        time_col="PROBLEM_DT_TM",
        target_col="PROBLEM_CODE",
        first_occurrence_only=True,
        match_type="exact",
    )

    # Validate pancreatitis picks min date (2020-10-01 instead of 2020-12-02)
    panc_date = res1[
        (res1["SUBJECT"] == 10001)
        & (res1["event_name"] == "charlson_pancreatitis")
    ]["first_occurrence_date"].values[0]
    assert panc_date == "2020-10-01", f"Expected 2020-10-01, got {panc_date}"
    print("✓ TEST 1 PASSED: Exact SNOMED match & min date aggregation works.")

    # TEST 2: ICD-10 Matching with Dot Stripping (1 dot, 2 dots, no dots)
    configs_icd = {
        "amnesia_event": {"codes": ["R413"]},  # Target code has no dots
        "limb_pain_event": {"codes": ["M79.60"]},  # Target code has 1 dot
        "corneal_event": {"codes": ["H1841"]},  # Target matches 2-dot source
    }

    res2 = build_historical_events_table(
        df=df_diagnoses,
        configs=configs_icd,
        subject_col="SUBJECT",
        time_col="DIAGNOSIS_DATE",
        target_col="DIAGNOSIS_CODE_ICD",
        first_occurrence_only=False,
        match_type="exact",
        strip_dots=True,
    )

    matched_subjects = set(res2["SUBJECT"].tolist())
    assert matched_subjects == {
        10001,
        10002,
        10003,
        10004,
        10005,
    }, f"Unexpected matches: {matched_subjects}"
    print(
        "✓ TEST 2 PASSED: Dot stripping matches formatted & unformatted ICD codes."
    )

    # TEST 3: ICD-10 Prefix Matching
    configs_prefix = {
        "m_codes": {"codes": ["M79"]}  # Should match M79.60 and M7960
    }

    res3 = build_historical_events_table(
        df=df_diagnoses,
        configs=configs_prefix,
        subject_col="SUBJECT",
        time_col="DIAGNOSIS_DATE",
        target_col="DIAGNOSIS_CODE_ICD",
        first_occurrence_only=False,
        match_type="prefix",
        strip_dots=True,
    )

    assert len(res3) == 2, f"Expected 2 matches for prefix M79, got {len(res3)}"
    print("✓ TEST 3 PASSED: ICD-10 prefix matching works.")

    # TEST 4: String Contains Matching (Prescriptions / Drugs)
    configs_drugs = {
        "abx_penicillin": {
            "keywords": ["amoxicillin"]
        },  # Case-insensitive substring
    }

    res4 = build_historical_events_table(
        df=df_prescriptions,
        configs=configs_drugs,
        subject_col="SUBJECT",
        time_col="ORDER_DT_TM",
        target_col="MEDICATION_NAME",
        first_occurrence_only=False,
        match_type="contains",
        case_sensitive=False,
    )

    assert len(res4) == 1, f"Expected 1 drug match, got {len(res4)}"
    assert res4.iloc[0]["SUBJECT"] == 10001
    print("✓ TEST 4 PASSED: Substring ('contains') matching on drug names works.")

    print("\n--- ALL TESTS PASSED SUCCESSFULLY ---")

In [52]:
# Run the tests
run_tests()

--- Running Test Bench ---

✓ TEST 1 PASSED: Exact SNOMED match & min date aggregation works.
✓ TEST 2 PASSED: Dot stripping matches formatted & unformatted ICD codes.
✓ TEST 3 PASSED: ICD-10 prefix matching works.
✓ TEST 4 PASSED: Substring ('contains') matching on drug names works.

--- ALL TESTS PASSED SUCCESSFULLY ---


In [53]:
def test_edge_cases():
    # Libraries
    import re

    print("--- Running Edge-Case Audit ---\n")

    # 1. Test SNOMED Integer vs String casting
    df_snomed = pd.DataFrame({
        "SUBJECT": [10001, 10002],
        "PROBLEM_CODE": [1077911000000105, "1077911000000105"], # int64 vs str
        "PROBLEM_DT_TM": pd.to_datetime(["2023-12-05", "2023-12-05"])
    })

    # Ensure conversion to clean string without trailing .0
    snomed_series = df_snomed["PROBLEM_CODE"].astype(str).str.replace(r'\.0$', '', regex=True)
    assert snomed_series.iloc[0] == "1077911000000105", "SNOMED float conversion error!"
    print("✓ Check 1 Passed: SNOMED long integers handled without scientific corruption.")

    # 2. Test Datetime Minimum Aggregation
    df_dates = pd.DataFrame({
        "SUBJECT": [10001, 10001],
        "PROBLEM_DT_TM": pd.to_datetime(["2022-11-28", "2020-01-15"]),
        "PROBLEM_CODE": ["A01", "A01"]
    })
    min_date = df_dates.groupby("SUBJECT")["PROBLEM_DT_TM"].min().iloc[0]
    assert min_date == pd.Timestamp("2020-01-15"), "Timestamp min aggregation failed!"
    print("✓ Check 2 Passed: Timestamp parsed and min date selected correctly.")

    # 3. Test Special Characters in Drug/Lab Search
    series_labs = pd.Series(["amoxycillin : >8", "Co-amoxiclav (500mg/125mg)"])
    target_keyword = "amoxycillin : >8"

    # Check regex escaping
    pattern = re.escape(target_keyword)
    matches = series_labs.str.contains(pattern, regex=True)
    assert matches.sum() == 1, "Special character regex escaping failed!"
    print("✓ Check 3 Passed: Special characters (>, :, parentheses) safely escaped.")

    # 4. Recursive YAML Code Extractor Test
    sample_yaml_config = {
        "kim_recent_antibiotics_or_class": {
            "kwargs": {
                "components": [
                    {"keywords": ["amoxicillin", "piperacillin"]},
                    {"keywords": ["antibiotics"]}
                ]
            }
        }
    }

    def extract_all_codes_recursive(config: dict) -> list:
        extracted = []
        if isinstance(config, dict):
            for k, v in config.items():
                if k in ("codes", "keywords"):
                    if isinstance(v, list):
                        extracted.extend(v)
                    elif isinstance(v, str):
                        extracted.append(v)
                else:
                    extracted.extend(extract_all_codes_recursive(v))
        elif isinstance(config, list):
            for item in config:
                extracted.extend(extract_all_codes_recursive(item))
        return list(set(extracted))

    extracted_terms = extract_all_codes_recursive(sample_yaml_config["kim_recent_antibiotics_or_class"])
    assert set(extracted_terms) == {"amoxicillin", "piperacillin", "antibiotics"}, "Recursive code extraction failed!"
    print("✓ Check 4 Passed: Complex nested YAML configurations parsed cleanly.")

    print("\n--- ALL EDGE CASE AUDITS PASSED ---")

In [54]:
test_edge_cases()

--- Running Edge-Case Audit ---

✓ Check 1 Passed: SNOMED long integers handled without scientific corruption.
✓ Check 2 Passed: Timestamp parsed and min date selected correctly.
✓ Check 3 Passed: Special characters (>, :, parentheses) safely escaped.
✓ Check 4 Passed: Complex nested YAML configurations parsed cleanly.

--- ALL EDGE CASE AUDITS PASSED ---


In [65]:
# Libraries
from typing import Dict, List, Any
from icare_risk.clinphen.utils.vectorized import (
    parse_phenotypes_to_batches,
    run_vectorized
)

domain_registry = {
    "prescribing": {
        "df": df_prescriptions,
        "subject_col": "SUBJECT",
        "time_col": "ORDER_DT_TM"
    },
    "problems": {
        "df": df_problems,
        "subject_col": "SUBJECT",
        "time_col": "PROBLEM_DT_TM"
    },
    "diagnoses": {
        "df": df_diagnoses,
        "subject_col": "SUBJECT",
        "time_col": "DIAGNOSIS_DATE"
    }
}

# Sample YAML dictionary reflecting your schema
sample_phenotypes_yaml = {
    "kim_recent_antibiotics_or_class": {
        "module": "icare_risk.clinphen.phenotypes.utils",
        "function": "derive_composite_rules",
        "domains": ["prescribing"],
        "kwargs": {
            "logic": "or",
            "components": [
                {
                    "extractor_type": "keyword",
                    "domain": "prescribing",
                    "text_col": "MEDICATION_NAME",
                    "keywords": ["amoxicillin", "piperacillin", "meropenem"],
                    "window": ["-2160h", "24h"],
                    "match_type": "contains",
                },
                {
                    "extractor_type": "keyword",
                    "domain": "prescribing",
                    "text_col": "THERAPEUTICAL_CLASS",
                    "keywords": ["antibiotics", "antihypertensives"],
                    "window": ["-720h", "0h"],  # Different window!
                    "match_type": "contains",
                },
            ],
        },
    },
    "tumbarello_hx_prior_esbl": {
        "module": "icare_risk.clinphen.phenotypes.utils",
        "function": "derive_composite_rules",
        "category": ["tumbarello_components"],
        "kwargs": {
            "logic": "max",
            "components": [
                {
                    "extractor_type": "history",
                    "domain": "problems",
                    "text_col": "PROBLEM_CODE",
                    "codes": ["235494005", "84089009"],
                    "window": None,  # Full historical timeline (no window)
                    "match_type": "exact",
                }
            ],
        },
    },
}

combined_df = run_vectorized(
    domains=domain_registry,
    phenotypes=sample_phenotypes_yaml
)


--- Running Vectorized Bench ---

--- 1. Parsing Phenotypes YAML ---
Generated 3 execution batch(es).

--- 2. Executing Batches ---

--- DEBUG BATCH 1 (Source 1) [prescribing -> MEDICATION_NAME] ---
Match Type: contains, Case Sensitive: False
YAML is looking for (first 5):
  - kim_recent_antibiotics_or_class__comp0
DataFrame top 5 most common values (with counts):
  - Amoxicillin 500mg: 1 occurrence(s)
  - Lisinopril 10mg : 1 occurrence(s)
  - Propofol 10mg/ml: 1 occurrence(s)
-> Matched 1 record(s) using column 'MEDICATION_NAME' with match_type='contains'.

--- DEBUG BATCH 2 (Source 1) [prescribing -> THERAPEUTICAL_CLASS] ---
Match Type: contains, Case Sensitive: False
YAML is looking for (first 5):
  - kim_recent_antibiotics_or_class__comp1
CRITICAL ERROR: Column 'THERAPEUTICAL_CLASS' does not exist in DataFrame!

--- DEBUG BATCH 3 (Source 1) [problems -> PROBLEM_CODE] ---
Match Type: exact, Case Sensitive: False
YAML is looking for (first 5):
  - tumbarello_hx_prior_esbl__comp0
Dat

In [56]:
import yaml
from icare_risk.clinphen.utils.vectorized import evaluate_temporal_windows
from icare_risk.utils.io import load_pkg_yaml

# Episodes / Cohort DataFrame
episodes_df = pd.DataFrame(
    [
        # Episode 1 (Subject 10001 - Admission 1):
        # Amoxicillin (2022-05-27) is ~5 days prior -> SHOULD MATCH 90d window (-2160h)
        # Diagnoses/Problems from 2020 are >1.5 years prior -> EXCLUDED from 90d, MATCHES lifetime
        {"EPISODE_ID": "EP_10001_A", "SUBJECT": 10001, "index_admission": "2022-06-01 08:00:00"},

        # Episode 2 (Subject 10001 - Admission 2 / Multi-episode test):
        # Lisinopril (2023-04-13) is ~18 days prior -> SHOULD MATCH 90d window
        # Amoxicillin (2022-05-27) is ~11 months prior -> EXCLUDED from 90d window
        {"EPISODE_ID": "EP_10001_B", "SUBJECT": 10001, "index_admission": "2023-05-01 10:00:00"},

        # Episode 3 (Subject 10002):
        # Propofol (2022-05-07) is 3 days prior -> SHOULD MATCH short windows
        {"EPISODE_ID": "EP_10002_A", "SUBJECT": 10002, "index_admission": "2022-05-10 14:00:00"},

        # Episode 4 (Subject 10003):
        # Diagnosis M7960 (2023-06-01) is 14 days prior
        {"EPISODE_ID": "EP_10003_A", "SUBJECT": 10003, "index_admission": "2023-06-15 09:00:00"},

        # Episode 5 (Subject 10004 - Leakage / Future event test):
        # Diagnosis H18.4.1 occurred on 2021-10-06, AFTER admission (2020-01-01) -> MUST BE EXCLUDED (0)
        {"EPISODE_ID": "EP_10004_A", "SUBJECT": 10004, "index_admission": "2020-01-01 00:00:00"},

        # Episode 6 (Subject 10005 - Exact boundary test):
        # Diagnosis R413 is on the exact same date ("2022-01-01") -> Tests strict vs. inclusive bounds
        {"EPISODE_ID": "EP_10005_A", "SUBJECT": 10005, "index_admission": "2022-01-01 00:00:00"},

        # Episode 7 (Subject 99999 - No history test):
        # Patient with no records in problems/diagnoses/prescriptions -> MUST BE ALL zeros
        {"EPISODE_ID": "EP_99999_A", "SUBJECT": 99999, "index_admission": "2023-01-01 12:00:00"},
    ]
)

# Convert to datetime
episodes_df["index_admission"] = pd.to_datetime(episodes_df["index_admission"])

# Load phenotypes config file
config = load_pkg_yaml('config/icare/phenotypes.yaml')

# 2. Run function directly passing the loaded YAML node
feature_df = evaluate_temporal_windows(
    episodes_df=episodes_df,
    events_df=combined_df,
    window_configs=sample_phenotypes_yaml,
    subject_col="SUBJECT",
    admission_col="index_admission",
    event_col="event_name",
    event_time_col="event_dt_tm",
    prefix="hx_"
)

print(feature_df)
print(feature_df.columns)

   EPISODE_ID  SUBJECT     index_admission  \
0  EP_10001_A    10001 2022-06-01 08:00:00   
1  EP_10001_B    10001 2023-05-01 10:00:00   
2  EP_10002_A    10002 2022-05-10 14:00:00   
3  EP_10003_A    10003 2023-06-15 09:00:00   
4  EP_10004_A    10004 2020-01-01 00:00:00   
5  EP_10005_A    10005 2022-01-01 00:00:00   
6  EP_99999_A    99999 2023-01-01 12:00:00   

   hx_kim_recent_antibiotics_or_class__comp0  \
0                                          1   
1                                          1   
2                                          0   
3                                          0   
4                                          0   
5                                          0   
6                                          0   

   hx_tumbarello_hx_prior_esbl__comp0  
0                                   1  
1                                   1  
2                                   0  
3                                   0  
4                                   0  
5     

In [64]:
from icare_risk.clinphen.utils.vectorized import aggregate_composite_phenotypes

final_df = aggregate_composite_phenotypes(
    df=feature_df,
    phenotype_configs=sample_phenotypes_yaml,
    prefix='hx_',
    drop_components=True
)
final_df = final_df.loc[:, ~final_df.columns.str.contains('__comp')]
print(final_df)

   EPISODE_ID  SUBJECT     index_admission  \
0  EP_10001_A    10001 2022-06-01 08:00:00   
1  EP_10001_B    10001 2023-05-01 10:00:00   
2  EP_10002_A    10002 2022-05-10 14:00:00   
3  EP_10003_A    10003 2023-06-15 09:00:00   
4  EP_10004_A    10004 2020-01-01 00:00:00   
5  EP_10005_A    10005 2022-01-01 00:00:00   
6  EP_99999_A    99999 2023-01-01 12:00:00   

   hx_kim_recent_antibiotics_or_class  hx_tumbarello_hx_prior_esbl  
0                                   1                            0  
1                                   1                            0  
2                                   0                            0  
3                                   0                            0  
4                                   0                            0  
5                                   0                            0  
6                                   0                            0  
